# Nested Supervised Training

This notebook trains a supervised target on the bundled Breast Cancer Wisconsin dataset. The example reshapes selected columns into a nested `measurements` array so the model sees repeated measurement objects rather than one wide flat row.

The imports mirror the training tutorial. The Breast Cancer records are already buffered as nested JSONL, so the notebook can focus on the schema.


In [1]:
import contextlib
import io
import logging

import lightning.pytorch as lit
import polars as pl
import torch
from loguru import logger
from rich.pretty import pprint

import json2vec as j2v

logger.remove()


Each record contains a list of measurement dictionaries plus a diagnosis label. This is intentionally small, but it demonstrates the pattern used for orders with line items, sessions with events, or entities with repeated attributes.


In [2]:
records = pl.read_ndjson("docs/data/breast-cancer.jsonl").head(32)

records.head()


measurements,diagnosis
list[struct[2]],str
"[{""mean_radius"",17.99}, {""mean_texture"",10.38}, … {""mean_smoothness"",0.1184}]","""malignant"""
"[{""mean_radius"",13.54}, {""mean_texture"",14.36}, … {""mean_smoothness"",0.09779}]","""benign"""
"[{""mean_radius"",20.57}, {""mean_texture"",17.77}, … {""mean_smoothness"",0.08474}]","""malignant"""
"[{""mean_radius"",13.08}, {""mean_texture"",15.71}, … {""mean_smoothness"",0.1075}]","""benign"""
"[{""mean_radius"",19.69}, {""mean_texture"",21.25}, … {""mean_smoothness"",0.1096}]","""malignant"""


The nested `Array` defines the measurement context. Inside that array, `name` identifies the measurement and `value` carries the numeric signal. The root-level `diagnosis` field is the supervised target.

In [3]:
model = j2v.Model.from_schema(
    j2v.Array(
        j2v.Category("name", query="[*].measurements[*].name", max_vocab_size=16),
        j2v.Number("value", query="[*].measurements[*].value"),
        name="measurements",
        max_length=5,
    ),
    j2v.Category("diagnosis", query="[*].diagnosis", target=True, max_vocab_size=2),
    d_model=16,
    n_layers=1,
    n_heads=4,
    batch_size=8,
    embed=True,
    optimizer=lambda module: torch.optim.AdamW(module.parameters(), lr=1e-2),
)


The data module does not need special nested-data code. The schema queries describe where values live, and the data module handles batching and tensorization.

In [4]:
datamodule = j2v.PolarsDataModule.from_model(
    model,
    train=records,
    validate=records,
    num_workers=0,
    persistent_workers=False,
    pin_memory=False,
    observation_buffer_size=32,
    chunk_batch_size=32,
    sample_rate=1.0,
)

Training here is intentionally minimal: the notebook proves the schema shape and supervised path, not benchmark performance.

In [5]:
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    logging.disable(logging.CRITICAL)
    try:
        lit.seed_everything(7, workers=True, verbose=False)
        trainer = lit.Trainer(
            accelerator="cpu",
            max_epochs=1,
            logger=False,
            enable_progress_bar=False,
            enable_model_summary=False,
            enable_checkpointing=False,
            limit_train_batches=1,
            limit_val_batches=1,
        )
        trainer.fit(model=model, datamodule=datamodule)
    finally:
        logging.disable(logging.NOTSET)

The plot is useful for nested schemas because it shows which modules belong to the root record and which belong to the repeated measurement context.

In [6]:
model.plot(detail=True)

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                      │
│  JSON2Vec                                                             d_model  16                                    │
│  Model                                                                 params  24,935                                │
│  Schema map                                                            arrays  2                                     │
│                                                                        fields  3                                     │
│                                                                       targets  1                                     │
│                                                                        embeds  1                                     │
│                               

Prediction decodes only configured targets. In this case, the output is the model response for `record/diagnosis`.

In [7]:
batch = [[record] for record in records.to_dicts()[:3]]
pprint(model.predict(batch))

{
│   'record/diagnosis': {
│   │   'state': {
│   │   │   'valued': [0.5146114230155945, 0.5143932104110718, 0.5147429704666138],
│   │   │   'null': [0.09393507987260818, 0.09385216236114502, 0.09382839500904083],
│   │   │   'padded': [0.10056453943252563, 0.1006062924861908, 0.10045140981674194],
│   │   │   'masked': [0.08014068752527237, 0.08016800135374069, 0.08016277849674225],
│   │   │   'other': [0.21074829995632172, 0.21098026633262634, 0.21081425249576569]
│   │   },
│   │   'content': {
│   │   │   'value': ['benign', 'benign', 'benign'],
│   │   │   'probability': [0.6467775106430054, 0.6472691297531128, 0.6472794413566589],
│   │   │   'topk': [[], [], []]
│   │   }
│   }
}